# 5 · Retrieval (all 4 configurations)

Runs the LangChain `HybridLegalRetriever` for each configuration over the 15 benchmark queries and saves the retrieved contexts to `results/contexts/`.

Unchunked configs use the original question; rechunked configs use the expanded query and add BM25 + cross-encoder re-ranking. **Machine 2** (needs local models).

In [1]:
import json
import pandas as pd
from config import config
from indian_marriage_legal_recommender.retrieval import get_retriever

df = pd.read_csv(config.BENCHMARK_EXPANDED_CSV)
config.CONTEXTS_DIR.mkdir(parents=True, exist_ok=True)

### The 4 retrieval configurations

Each config pairs an embedding model with a chunking/re-ranking strategy. Unchunked configs do plain semantic top-k on the original question; rechunked configs retrieve with the **expanded query** and add BM25 + cross-encoder re-ranking followed by a parent-child merge.

In [2]:
from IPython.display import display

prof_df = pd.DataFrame([
    {
        'config': k,
        'embed_model': p.embed_model.split('/')[-1],
        'dim': p.dim,
        'chunked': p.chunked,
        'query': 'expanded' if p.use_expanded_query else 'original',
        'hybrid_rerank': p.hybrid,
    }
    for k, p in config.PROFILES.items()
])
print(f'{len(prof_df)} configs × {len(df)} benchmark queries = {len(prof_df) * len(df)} retrievals')
display(prof_df)

4 configs × 15 benchmark queries = 60 retrievals


,config,embed_model,dim,chunked,query,hybrid_rerank
0,jina_unchunked,jina-embeddings-v3,1024,False,original,False
1,baseline_unchunked,all-MiniLM-L6-v2,384,False,original,False
2,jina_rechunked,jina-embeddings-v3,1024,True,expanded,True
3,baseline_rechunked,all-MiniLM-L6-v2,384,True,expanded,True


In [4]:
n = len(df)
for key, prof in config.PROFILES.items():
    retriever = get_retriever(key, top_k_final=config.TOP_K_FINAL)
    strategy = 'hybrid re-rank' if prof.hybrid else 'semantic top-k'
    query_kind = 'expanded query' if prof.use_expanded_query else 'original question'
    print(f'\n▶ {key}  ({prof.embed_model.split("/")[-1]}, {strategy}, {query_kind})')

    out = []
    for i, row in df.iterrows():
        query = row['Expanded_Query'] if prof.use_expanded_query else row['Question']
        docs = retriever.invoke(query)
        out.append({'id': int(i), 'query': row['Question'],
                    'contexts': [d.page_content for d in docs]})

        # show which sections were retrieved for this query
        print(f'  [{i + 1}/{n}] {row["Question"]}')
        for d in docs:
            print(f'     → {d.metadata.get("section", "?")}')

    path = config.CONTEXTS_DIR / f'{key}_contexts.json'
    path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'  Saved {path.relative_to(config.ROOT_DIR)}')


▶ jina_unchunked  (jina-embeddings-v3, semantic top-k, original question)
  [1/15] I am a Sikh man planning to get married in Punjab. Does my marriage need to be registered under the Hindu Marriage Act?
     → The Jharkhand Compulsory Registration of Marriages Act, 2017 | Section 23: Time Limit of Compulsory Registration of Marriage.
     → The Hindu Marriage Act, 1955 | Section 8: Registration of Hindu marriages.-
     → The Jharkhand Compulsory Registration of Marriages Act, 2017 | Section 11: Compulsory registration of marriages.
     → Uttarakhand Compulsory Registration of Marriage Act, 2010 | Section 18: Savings.
     → The Orissa Muhammedan Marriages and Divorces Registration Act, 1949 | Section 26: Savings.
  [2/15] As a Muslim woman in Uttar Pradesh, if my husband pronounces 'Talaq' three times in one sitting, is my marriage legally terminated?
     → The Muslim Women (protection of Rights on Marriage) Act, 2019 | Section 4: Punishment for pronouncing talaq.
     → The Muslim

### Same question, 4 configs side by side

The whole point of this notebook is comparing configurations — here is what each retrieved for a single benchmark question.

In [5]:
QUERY_ID = 0   # which benchmark question to compare (0–14)

loaded = {
    key: json.loads((config.CONTEXTS_DIR / f'{key}_contexts.json').read_text())
    for key in config.PROFILES
}
question = loaded[list(config.PROFILES)[0]][QUERY_ID]['query']
print(f'Q{QUERY_ID}: {question}\n' + '=' * 80)

for key in config.PROFILES:
    contexts = loaded[key][QUERY_ID]['contexts']
    print(f'\n▶ {key}  ({len(contexts)} contexts)')
    for c in contexts:
        header = c.split('\n', 1)[0]                     # first line = section header
        body = c.split('\n', 1)[1] if '\n' in c else ''
        snippet = ' '.join(body.split())[:200]
        print(f'   • {header}')
        print(f'     {snippet}...')

Q0: I am a Sikh man planning to get married in Punjab. Does my marriage need to be registered under the Hindu Marriage Act?

▶ jina_unchunked  (5 contexts)
   • The Jharkhand Compulsory Registration of Marriages Act, 2017 | Section 23: Time Limit of Compulsory Registration of Marriage.
     - After the commencement of this Act, it shall be compulsory for all people mentioned in Section 10, to get their marriage registered in the office of Marriage Registrar of that particular jurisdictio...
   • The Hindu Marriage Act, 1955 | Section 8: Registration of Hindu marriages.-
     (1) For the purpose of facilitating the proof of Hindu marriages, the State Government may make rules providing that the parties to any such marriage may have the particulars relating to their marriag...
   • The Jharkhand Compulsory Registration of Marriages Act, 2017 | Section 11: Compulsory registration of marriages.
     (1) After the commencement of this Act, it shall be compulsory for all citizen residing in 